In [ ]:
# | default_exp features.wps_genomewide

In [ ]:
# | export
from __future__ import annotations
import pandas as pd
import numpy as np
import structlog
from kreview.eval_engine import FeatureEvaluator, parse_array

log = structlog.get_logger()

In [ ]:
# | export
def _to_array(val) -> np.ndarray | None:
    """Convert a parquet array value to numpy, handling all storage formats.

    Handles:
    - numpy arrays (native parquet ``list<float>``, DuckDB ``FLOAT[]``)
    - Python lists/tuples (e.g. from pandas object columns)
    - String-encoded arrays (legacy format: ``"[1.0 2.0 3.0]"``)
    - None / NaN → returns None

    Returns:
        numpy array of floats, or None if the value is empty/invalid.
    """
    if val is None:
        return None
    if isinstance(val, np.ndarray):
        return val.astype(float) if val.size > 0 else None
    if isinstance(val, (list, tuple)):
        return np.array(val, dtype=float) if len(val) > 0 else None
    if isinstance(val, str):
        parsed = parse_array(val)
        return np.array(parsed, dtype=float) if parsed else None
    # Scalar NaN check
    try:
        if pd.isna(val):
            return None
    except (TypeError, ValueError):
        pass
    log.warning("wpsgenome_unexpected_type", val_type=type(val).__name__)
    return None


In [ ]:
# | export


class WPSGenomeEvaluator(FeatureEvaluator):
    """Extracts genome-wide WPS metrics per region_type.

    For each WPS array column (wps_nuc, wps_tf, prot_frac_nuc, prot_frac_tf)
    and each region_type (TSS, CTCF), computes aggregate statistics across
    all genomic regions of that type:

    - mean: average of per-region array means (signal level)
    - peak_valley: average of per-region (max - min) amplitude
    - std: standard deviation of per-region array means (variability)
    - mad: median absolute deviation of per-region array means (robust dispersion)

    Output features (example): ``TSS_wps_nuc_mean``, ``CTCF_prot_frac_tf_peak_valley``

    Extraction paths:
        - **SQL pushdown** (``extract_sql()``): Uses DuckDB native ``list_avg``,
          ``list_max``, ``list_min`` to aggregate 1.9B rows → ~158 rows inside
          DuckDB. MAD is computed via a lightweight second query (exact).
          Memory: <1 GB vs 96+ GB for the Python path.
        - **Python fallback** (``extract()``): Per-sample chunked extraction
          with numpy — used when SQL pushdown is unavailable or fails.

    Note:
        Krewlyzer stores WPS arrays as native ``list<float>`` in parquet,
        not as string-encoded arrays. The ``extract()`` method handles
        numpy arrays directly via ``_to_array()`` — no string parsing needed.

    Note:
        FFT spectral features were evaluated and found to be redundant with
        basic stats (r=0.91–1.00 with peak_valley/std). Nucleosome periodicity
        is captured at the chromosome level by ``WPSBackgroundEvaluator``
        (NRL, periodicity_score). See ``wps_fft_analysis.md`` for details.
    """

    name = "WPSGenome"
    source_file = ".WPS.parquet"
    tier = 2
    category = "epigenetics_and_geometry"
    # Column projection: only read the 5 columns used by extract().
    # WPS parquets have many additional columns — loading all of them
    # caused OOM on genome-wide runs (~30K rows × 50+ cols per sample).
    extract_columns = [
        "region_type",
        "wps_nuc",
        "wps_tf",
        "prot_frac_nuc",
        "prot_frac_tf",
    ]
    max_chunk_rows = 2_000_000  # Aggressive chunking for genomewide WPS data

    # Array column names used by extract() and extract_sql()
    _array_cols = ("wps_nuc", "wps_tf", "prot_frac_nuc", "prot_frac_tf")

    # SQL pivot: multi-row results keyed by region_type → CLI pivots to wide format
    sql_pivot_column = "region_type"

    def extract_sql(self) -> str | None:
        """DuckDB SQL pushdown for genome-wide WPS extraction.

        Uses native list functions (list_avg, list_max, list_min) to aggregate
        1.9B rows → ~158 rows inside DuckDB. Memory: <1 GB vs 96+ GB.

        Returns multi-row result (one per sample × region_type). The CLI
        pivots this to wide format matching extract() output naming.

        Note: MAD is NOT computed here — it's added by a lightweight Python
        post-step using a second DuckDB query on row-level means (exact).
        """
        return """
            WITH per_region AS (
                SELECT
                    regexp_extract(filename, '/([^/]+)/[^/]+$', 1) AS sample_id,
                    REPLACE(REPLACE(REPLACE(CAST(region_type AS VARCHAR),
                        ' ', '_'), '-', '_'), '|', '_') AS region_type,
                    list_avg(wps_nuc) AS wps_nuc_row_mean,
                    (list_max(wps_nuc) - list_min(wps_nuc)) AS wps_nuc_row_pv,
                    list_avg(wps_tf) AS wps_tf_row_mean,
                    (list_max(wps_tf) - list_min(wps_tf)) AS wps_tf_row_pv,
                    list_avg(prot_frac_nuc) AS prot_frac_nuc_row_mean,
                    (list_max(prot_frac_nuc) - list_min(prot_frac_nuc))
                        AS prot_frac_nuc_row_pv,
                    list_avg(prot_frac_tf) AS prot_frac_tf_row_mean,
                    (list_max(prot_frac_tf) - list_min(prot_frac_tf))
                        AS prot_frac_tf_row_pv
                FROM read_parquet(
                    ?,
                    filename=true,
                    union_by_name=true,
                    hive_partitioning=false
                )
                WHERE region_type IS NOT NULL
            )
            SELECT
                sample_id,
                region_type,
                -- mean: average of per-region array means
                AVG(wps_nuc_row_mean)       AS wps_nuc_mean,
                AVG(wps_tf_row_mean)        AS wps_tf_mean,
                AVG(prot_frac_nuc_row_mean) AS prot_frac_nuc_mean,
                AVG(prot_frac_tf_row_mean)  AS prot_frac_tf_mean,
                -- peak_valley: average of per-region (max - min)
                AVG(wps_nuc_row_pv)         AS wps_nuc_peak_valley,
                AVG(wps_tf_row_pv)          AS wps_tf_peak_valley,
                AVG(prot_frac_nuc_row_pv)   AS prot_frac_nuc_peak_valley,
                AVG(prot_frac_tf_row_pv)    AS prot_frac_tf_peak_valley,
                -- std: std of per-region array means (variability)
                STDDEV(wps_nuc_row_mean)       AS wps_nuc_std,
                STDDEV(wps_tf_row_mean)        AS wps_tf_std,
                STDDEV(prot_frac_nuc_row_mean) AS prot_frac_nuc_std,
                STDDEV(prot_frac_tf_row_mean)  AS prot_frac_tf_std
            FROM per_region
            GROUP BY sample_id, region_type
        """

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        """Extract WPS features aggregated per region_type.

        For each region_type (TSS, CTCF) and each array column, computes:

        - mean: average of per-region array means (signal level)
        - peak_valley: average of per-region (max - min) (nucleosome amplitude)
        - std: std of per-region array means (variability across regions)
        - mad: MAD of per-region array means (robust dispersion)

        Handles both numpy arrays (from native parquet ``list<float>``)
        and string-encoded arrays (legacy format, via ``parse_array()``).
        """
        if df.empty:
            return {}

        cols = set(df.columns)
        if "region_type" not in cols:
            log.warning(
                "wpsgenome_missing_region_type",
                evaluator=self.name,
                columns=sorted(cols),
            )
            return {}

        extracted: dict[str, float] = {}
        n_regions_processed = 0

        for rt in df["region_type"].unique():
            if rt is None:
                continue
            rt_clean = str(rt).replace(" ", "_").replace("-", "_").replace("|", "_")
            rt_df = df[df["region_type"] == rt]

            for a in self._array_cols:
                if a not in cols:
                    continue

                # Collect per-region array statistics.
                # Data can be numpy arrays (native parquet list<float>)
                # or strings (legacy format).
                row_means: list[float] = []
                row_pvs: list[float] = []

                for val in rt_df[a]:
                    arr = _to_array(val)
                    if arr is not None and len(arr) > 0:
                        row_means.append(float(np.mean(arr)))
                        row_pvs.append(float(np.max(arr) - np.min(arr)))

                if not row_means:
                    log.debug(
                        "wpsgenome_empty_column",
                        region_type=rt_clean,
                        column=a,
                        n_rows=len(rt_df),
                    )
                    continue

                means_arr = np.array(row_means)
                extracted[f"{rt_clean}_{a}_mean"] = float(np.mean(means_arr))
                extracted[f"{rt_clean}_{a}_peak_valley"] = float(np.mean(row_pvs))
                extracted[f"{rt_clean}_{a}_std"] = float(np.std(means_arr))
                extracted[f"{rt_clean}_{a}_mad"] = float(
                    np.median(np.abs(means_arr - np.median(means_arr)))
                )
                n_regions_processed += len(row_means)

        if not extracted:
            log.warning(
                "wpsgenome_no_features_extracted",
                evaluator=self.name,
                n_rows=len(df),
                region_types=df["region_type"].unique().tolist(),
            )
        else:
            log.debug(
                "wpsgenome_extracted",
                n_features=len(extracted),
                n_regions=n_regions_processed,
            )

        return extracted

In [ ]:
# | test
# Test WPSGenome extract with numpy arrays (native parquet format)
evaluator = WPSGenomeEvaluator()

# Test 1: numpy array input (how DuckDB loads native list<float>)
df = pd.DataFrame([{
    "region_type": "TSS",
    "wps_nuc": np.array([1.0, 2.0, 3.0]),
    "wps_tf": np.array([4.0, 5.0]),
    "prot_frac_nuc": np.array([0.1, 0.2]),
    "prot_frac_tf": np.array([0.3, 0.4]),
}])
res = evaluator.extract(df)
assert "TSS_wps_nuc_mean" in res
assert abs(res["TSS_wps_nuc_mean"] - 2.0) < 1e-6

# Test 2: string array input (legacy format)
df2 = pd.DataFrame([{
    "region_type": "Promo",
    "wps_nuc": "[1.0 1.0]",
}])
res2 = evaluator.extract(df2)
assert res2["Promo_wps_nuc_mean"] == 1.0

# Test 3: empty DataFrame returns empty dict
assert evaluator.extract(pd.DataFrame()) == {}

print(f"WPSGenome tests passed: {len(res)} features from numpy, {len(res2)} from string")